In [41]:
import os
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf

from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [42]:
BASE_DIR = "smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR   = f"{BASE_DIR}/classification/val"
TEST_DIR  = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

print("SETUP COMPLETE")

SETUP COMPLETE


In [43]:
# IMAGE SETTINGS

IMG_SIZE  = (224, 224)
BATCH     = 32
EPOCHS    = 15
NUM_CLASSES = 26

ftrain_datagen = ImageDataGenerator(

    preprocessing_function=preprocess_input,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.15,

    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# =========================================================
# LOAD DATASETS
# =========================================================

train_gen = train_datagen.flow_from_directory(

    TRAIN_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(

    VAL_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(

    TEST_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH,

    class_mode='categorical',

    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())

print(f"\nNUMBER OF CLASSES: {len(CLASS_NAMES)}")

print(CLASS_NAMES)

Found 1820 images belonging to 26 classes.
Found 390 images belonging to 26 classes.
Found 390 images belonging to 26 classes.

NUMBER OF CLASSES: 26
['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'train', 'truck']


In [44]:
# LOAD PRETRAINED RESNET50
# =========================================================

base = ResNet50(

    weights='imagenet',

    include_top=False,

    input_shape=(224, 224, 3)
)

print("\nPRETRAINED RESNET50 LOADED")


PRETRAINED RESNET50 LOADED


In [45]:
# # # Freeze all layers
# base.trainable = False

# # # Optional fine-tuning: unfreeze last N layers
# unfreeze_last = 0

# if unfreeze_last > 0:
#     for layer in base.layers[-unfreeze_last:]:
#         layer.trainable = False

# # for layer in base.layers[:-20]:
#     # layer.trainable = False

# FREEZE BACKBONE
# =========================================================

base.trainable = False

In [46]:
# # Custom classification head
x = base.output

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.5)(x)

x = layers.Dense(256, activation='relu')(x)

x = layers.Dropout(0.3)(x)

output = layers.Dense(

    NUM_CLASSES,

    activation='softmax'
)(x)



In [47]:
model = models.Model(

    inputs=base.input,

    outputs=output,

    name='ResNet50'
)

In [48]:
# COMPILE MODEL
# =========================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(1e-4),

    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.1
    ),

    metrics=[

        'accuracy',

        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name='top5_acc'
        )
    ]
)

print("\nMODEL COMPILED")



MODEL COMPILED


In [49]:
# CALLBACKS
# =========================================================

callbacks_list = [

    callbacks.ModelCheckpoint(

        'models/ResNet50_best.keras',

        save_best_only=True,

        monitor='val_accuracy',

        verbose=1
    ),

    callbacks.EarlyStopping(

        monitor='val_accuracy',

        patience=5,

        restore_best_weights=True,

        verbose=1
    ),

    callbacks.ReduceLROnPlateau(

        monitor='val_loss',

        factor=0.5,

        patience=2,

        min_lr=1e-7,

        verbose=1
    )
]


In [50]:
model.summary()

Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_4[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,127,130 (92.04 MB)

 Trainable params: 535,322 (2.04 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

In [51]:
# INITIAL TRAINING
# =========================================================

print("\nSTARTING INITIAL TRAINING")

history = model.fit(

    train_gen,

    validation_data=val_gen,

    epochs= EPOCHS,

    callbacks=callbacks_list
)



STARTING INITIAL TRAINING
Epoch 1/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0410 - loss: 4.8065 - top5_acc: 0.2394
Epoch 1: val_accuracy improved from None to 0.15385, saving model to models/ResNet50_best.keras

Epoch 1: finished saving model to models/ResNet50_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 190s 3s/step - accuracy: 0.0610 - loss: 4.5675 - top5_acc: 0.2819 - val_accuracy: 0.1538 - val_loss: 3.0754 - val_top5_acc: 0.4744 - learning_rate: 1.0000e-04
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1483 - loss: 3.7532 - top5_acc: 0.4267
Epoch 2: val_accuracy improved from 0.15385 to 0.33590, saving model to models/ResNet50_best.keras

Epoch 2: finished saving model to models/ResNet50_best.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 160s 3s/step - accuracy: 0.1912 - loss: 3.5493 - top5_acc: 0.4698 - val_accuracy: 0.3359 - val_loss: 2.6698 - val_top5_acc: 0.5974 - learning_rate: 1.0000e-04
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.2733 - loss: 2.

In [52]:
model.evaluate(test_gen)

13/13 ━━━━━━━━━━━━━━━━━━━━ 33s 3s/step - accuracy: 0.3590 - loss: 2.5022 - top5_acc: 0.6667


[2.5021820068359375, 0.3589743673801422, 0.6666666865348816]